In [ ]:
import json
import os
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import supervision as sv
from PIL import Image
from rfdetr import RFDETRBase
from supervision.metrics import MeanAveragePrecision
from tqdm import tqdm

In [ ]:
# Set Roboflow API Key
os.environ['ROBOFLOW_API_KEY'] = ""

# Define dataset paths - UPDATE THESE PATHS
DATASET_NAME = "underwater_COCO_dataset"

# ── Paths ─────────────────────────────────────────────────────────────────────
DATASET_ROOT = Path("path/to/dataset")          # root containing train/ valid/ test/
OUTPUT_DIR   = Path("path/to/output/models")    # where checkpoints are saved
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Training hyperparameters ──────────────────────────────────────────────────
EPOCHS           = 50
BATCH_SIZE       = 4
GRAD_ACCUM_STEPS = 16
LEARNING_RATE    = 1e-4

# ── Inference ─────────────────────────────────────────────────────────────────
CONFIDENCE_THRESHOLD = 0.5

print("✓ Configuration loaded")
print(f"  Dataset root : {DATASET_ROOT}")
print(f"  Output dir   : {OUTPUT_DIR}")
print(f"  Epochs       : {EPOCHS}  |  Batch size: {BATCH_SIZE}  |  LR: {LEARNING_RATE}")


In [ ]:
# Define categories for the aquatic dataset
CATEGORIES = [
    {"id": 0, "name": "fish",        "supercategory": "animal"},
    {"id": 1, "name": "jellyfish",   "supercategory": "animal"},
    {"id": 2, "name": "penguin",     "supercategory": "animal"},
    {"id": 3, "name": "puffer_fish", "supercategory": "animal"},
    {"id": 4, "name": "shark",       "supercategory": "animal"},
    {"id": 5, "name": "stingray",    "supercategory": "animal"},
    {"id": 6, "name": "starfish",    "supercategory": "animal"},
]

print(f"Dataset: Aquatic Animals  |  {len(CATEGORIES)} classes\n")
for cat in CATEGORIES:
    print(f"  [{cat['id']}] {cat['name']}")

In [ ]:
def convert_yolo_to_coco(images_dir: Path, labels_dir: Path,
                         output_dir: Path, categories: list) -> dict:
    coco = {
        "info": {}, "licenses": [],
        "categories": categories,
        "images": [], "annotations": [],
    }
    ann_id = img_id = 0

    for img_file in sorted(os.listdir(images_dir)):
        if not img_file.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        try:
            img = Image.open(images_dir / img_file)
            w, h = img.size
        except Exception as e:
            print(f"  ⚠ Skipping {img_file}: {e}")
            continue

        coco["images"].append({"id": img_id, "file_name": img_file, "width": w, "height": h})

        label_path = labels_dir / (Path(img_file).stem + ".txt")
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:5])
                x_min = int((cx - bw / 2) * w)
                y_min = int((cy - bh / 2) * h)
                box_w = int(bw * w)
                box_h = int(bh * h)
                coco["annotations"].append({
                    "id": ann_id,
                    "image_id": img_id,
                    "category_id": cls,
                    "bbox": [x_min, y_min, box_w, box_h],
                    "area": box_w * box_h,
                    "iscrowd": 0,
                })
                ann_id += 1
        img_id += 1

    out_path = output_dir / "_annotations.coco.json"
    out_path.write_text(json.dumps(coco, indent=2))
    print(f"  ✓ {img_id} images  |  {ann_id} annotations  →  {out_path}")
    return coco


splits = {"train": "Training", "valid": "Validation", "test": "Test"}

for split, label in splits.items():
    print(f"{label} set …")
    convert_yolo_to_coco(
        images_dir=DATASET_ROOT / split / "images",
        labels_dir=DATASET_ROOT / split / "labels",
        output_dir=DATASET_ROOT / split,
        categories=CATEGORIES,
    )

print("\n✅ All splits converted to COCO format.")


In [ ]:
model = RFDETRBase()
history = []

def _on_epoch_end(data: dict):
    history.append(data)
    print(
        f"  Epoch {data.get('epoch', 0):>3}  |  "
        f"Train loss: {data.get('train_loss', 0):.4f}  |  "
        f"Val loss: {data.get('test_loss', 0):.4f}  |  "
        f"mAP: {data.get('test_map', 0):.4f}"
    )

model.callbacks["on_fit_epoch_end"].append(_on_epoch_end)
print("Model initialised — starting training …\n")

model.train(
    dataset_dir=str(DATASET_ROOT),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    lr=LEARNING_RATE,
    output_dir=str(OUTPUT_DIR),
    use_ema=True,
    tensorboard=True,
    early_stopping=True,
    early_stopping_patience=10,
    amp=True,
    device="cuda",
    num_workers=0,
    multi_scale=False,
    resolution=448,
)

print(f"\n✅ Training complete  |  Checkpoints → {OUTPUT_DIR}")


In [ ]:
df = pd.DataFrame(history)
print(f"Epochs recorded: {len(df)}")
print(df[["epoch", "train_loss", "test_loss", "test_map"]].tail(5).to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

ax = axes[0, 0]
ax.plot(df["epoch"], df["train_loss"], label="Train", marker="o", linewidth=2)
ax.plot(df["epoch"], df["test_loss"], label="Validation", marker="o", linewidth=2, linestyle="--")
ax.set_title("Loss", fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
if "test_map" in df.columns:
    ax.plot(df["epoch"], df["test_map"], label="mAP@0.5:0.95", color="#2ecc71", marker="o", linewidth=2)
ax.set_title("Validation mAP", fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("mAP")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
if "lr" in df.columns:
    ax.plot(df["epoch"], df["lr"], label="Learning Rate", color="#e67e22", marker="o", linewidth=2)
ax.set_title("Learning Rate Schedule", fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("LR")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
if "test_map" in df.columns and "test_map_ema" in df.columns:
    ax.plot(df["epoch"], df["test_map"], label="Regular mAP", marker="o", linewidth=2)
    ax.plot(df["epoch"], df["test_map_ema"], label="EMA mAP", marker="s", linewidth=2, linestyle="--")
    ax.set_title("Regular vs EMA mAP", fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("mAP")
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.set_visible(False)

fig.suptitle("RF-DETR Baseline — Training Metrics (Aquatic Dataset)", fontsize=14, fontweight="bold")
plt.tight_layout()
save_path = OUTPUT_DIR / "training_curves.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Saved → {save_path}")


In [ ]:
CHECKPOINT = OUTPUT_DIR / "checkpoint_best_total.pth"

model = RFDETRBase(pretrain_weights=str(CHECKPOINT))
model.optimize_for_inference()

print(f"✓ Model loaded from: {CHECKPOINT}")

ds = sv.DetectionDataset.from_coco(
    images_directory_path=str(DATASET_ROOT / "test"),
    annotations_path=str(DATASET_ROOT / "test" / "_annotations.coco.json"),
)
print(f"✓ Test set loaded: {len(ds)} images  |  Classes: {ds.classes}")

In [ ]:
idx = random.randint(0, len(ds) - 1)
path, _, annotations = ds[idx]
image = Image.open(path).convert("RGB")

detections = model.predict(image, threshold=CONFIDENCE_THRESHOLD)

print(f"Image  : {Path(path).name}")
print(f"GT     : {len(annotations)} objects")
print(f"Pred   : {len(detections)} detections  (threshold={CONFIDENCE_THRESHOLD})")

text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)
thickness = sv.calculate_optimal_line_thickness(resolution_wh=image.size)
palette = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff66ff", "#3399ff",
    "#ff66b2", "#ff8080", "#b266ff",
])

bbox_ann = sv.BoxAnnotator(color=palette, thickness=thickness)
label_ann = sv.LabelAnnotator(
    color=palette,
    text_color=sv.Color.BLACK,
    text_scale=text_scale,
    text_thickness=thickness,
)

gt_labels = [ds.classes[c] for c in annotations.class_id]
pred_labels = [f"{ds.classes[c]} {conf:.2f}" for c, conf in zip(detections.class_id, detections.confidence)]

gt_img = label_ann.annotate(bbox_ann.annotate(image.copy(), annotations), annotations, gt_labels)
pred_img = label_ann.annotate(bbox_ann.annotate(image.copy(), detections), detections, pred_labels)

sv.plot_images_grid(
    images=[gt_img, pred_img],
    grid_size=(1, 2),
    titles=["Ground Truth", f"RF-DETR  (conf ≥ {CONFIDENCE_THRESHOLD})"],
)

In [ ]:
targets, predictions = [], []

for path, _, annotations in tqdm(ds, desc="Evaluating test set"):
    image = Image.open(path).convert("RGB")
    detections = model.predict(image, threshold=0)
    targets.append(annotations)
    predictions.append(detections)

metric = MeanAveragePrecision()
metric.update(predictions, targets)
map_result = metric.compute()

print("\n" + "=" * 45)
print("TEST SET RESULTS — Aquatic Baseline")
print("=" * 45)
print(f"  mAP@0.5:0.95 : {map_result.map50_95:.4f}")
print(f"  mAP@0.5      : {map_result.map50:.4f}")
print(f"  mAP@0.75     : {map_result.map75:.4f}")


In [ ]:
SOURCE_VIDEO = Path("path/to/input_video.mp4")
OUTPUT_VIDEO = OUTPUT_DIR / "aquatic_detections.mp4"

def annotate_frame(frame, _index):
    detections = model.predict(frame, threshold=CONFIDENCE_THRESHOLD)
    labels = [
        f"{CATEGORIES[c]['name']} {conf:.2f}"
        for c, conf in zip(detections.class_id, detections.confidence)
    ]
    frame = sv.BoxAnnotator().annotate(frame, detections)
    frame = sv.LabelAnnotator().annotate(frame, detections, labels)
    return frame

if SOURCE_VIDEO.exists():
    sv.process_video(
        source_path=str(SOURCE_VIDEO),
        target_path=str(OUTPUT_VIDEO),
        callback=annotate_frame,
    )
    print(f"✅ Annotated video saved → {OUTPUT_VIDEO}")
else:
    print(f"⚠ Source video not found: {SOURCE_VIDEO}")
    print("  Update SOURCE_VIDEO path and re-run this cell.")
